#### Construcción del pipeline usando las decisiones de preprocesamiento del cuaderno 01

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    StandardScaler,
)

In [ ]:
DATA_PATH = "../data/raw/HousingData.csv"

df = pd.read_csv(DATA_PATH)

df.head()

In [ ]:
target = "MEDV"

X = df.drop(columns=[target])
y = df[target]

X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print(X_train.shape)
print(X_test.shape)

In [ ]:
def build_pipeline():

    log_transform_cols = ["CRIM", "ZN"]

    median_cols = [
        "INDUS",
        "AGE",
        "LSTAT",
    ]

    mode_cols = ["CHAS"]

    other_numeric_cols = [
        "NOX",
        "RM",
        "DIS",
        "RAD",
        "TAX",
        "PTRATIO",
        "B",
    ]

    log_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median"),
            ),
            (
                "log",
                FunctionTransformer(
                    np.log1p,
                    feature_names_out="one-to-one",
                ),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ]
    )

    median_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median"),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ]
    )

    mode_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                ),
            ),
        ]
    )

    other_numeric_transformer = Pipeline(
        steps=[
            (
                "scaler",
                StandardScaler(),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "log",
                log_transformer,
                log_transform_cols,
            ),
            (
                "median",
                median_transformer,
                median_cols,
            ),
            (
                "mode",
                mode_transformer,
                mode_cols,
            ),
            (
                "other_num",
                other_numeric_transformer,
                other_numeric_cols,
            ),
        ],
        remainder="drop",
    )

    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42,
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    return pipeline


pipeline = build_pipeline()

pipeline

In [ ]:
pipeline.fit(X_train, y_train)

print("Training complete.")

In [ ]:
predictions = pipeline.predict(X_test)

predictions[:10]

In [ ]:
mae = mean_absolute_error(
    y_test,
    predictions,
)

rmse = mean_squared_error(
    y_test,
    predictions,
) ** 0.5

r2 = r2_score(
    y_test,
    predictions,
)

print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R2:   {r2:.4f}")

In [ ]:
results = pd.DataFrame({
    "actual": y_test,
    "predicted": predictions,
    "error": y_test - predictions,
})

results.head(10)

In [ ]:
plt.figure(figsize=(7, 6))

plt.scatter(
    y_test,
    predictions,
    alpha=0.7,
)

plt.xlabel("Actual MEDV")
plt.ylabel("Predicted MEDV")

plt.title(
    "Actual vs Predicted Housing Prices"
)

plt.tight_layout()
plt.show()

In [ ]:
feature_names = pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": pipeline.named_steps[
        "model"
    ].feature_importances_,
}).sort_values(
    "importance",
    ascending=False,
)

feature_importance

In [ ]:
feature_importance.plot(
    x="feature",
    y="importance",
    kind="bar",
    figsize=(12, 6),
)

plt.title("Feature Importance")
plt.ylabel("Importance")

plt.tight_layout()
plt.show()

In [ ]:
MODEL_PATH = Path(
    "../models/housing_model.joblib"
)

MODEL_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

joblib.dump(
    pipeline,
    MODEL_PATH,
)

print(f"Saved to {MODEL_PATH}")

In [ ]:
loaded_model = joblib.load(MODEL_PATH)

sample = X_test.iloc[[0]]

prediction = loaded_model.predict(sample)[0]

print("Prediction:", prediction)
print("Actual:", y_test.iloc[0])

sample